In [ ]:
import os
import netCDF4 as nc
import numpy as np
import pandas as pd
import cftime

def read_netCDF(file_path):
    with nc.Dataset(file_path, 'r') as dataset:
        print(f"Variables in {os.path.basename(file_path)}: {list(dataset.variables.keys())}")


directory = '/Users/aymanelyoussoufi/Documents/RESEARCH/'
enso_df = pd.read_csv(os.path.join(directory, 'ENSO.csv'))

#loop
for filename in os.listdir(directory):
    if filename.startswith('chirps'):
        file_path = os.path.join(directory, filename)
        read_netCDF(file_path)

Variables in chirps_precip_daily.1981-01-01.2023-12-31.NM.nc: ['latitude', 'longitude', 'time', 'lon', 'dayofyear', 'precip']
Variables in chirps_precip_daily.1981-01-01.2023-12-31.WV.nc: ['latitude', 'longitude', 'time', 'lon', 'dayofyear', 'precip']
Variables in chirps_precip_daily.1981-01-01.2023-12-31.MI.nc: ['latitude', 'longitude', 'time', 'lon', 'dayofyear', 'precip']
Variables in chirps_precip_daily.1981-01-01.2023-12-31.IL.nc: ['latitude', 'longitude', 'time', 'lon', 'dayofyear', 'precip']
Variables in chirps_precip_daily.1981-01-01.2023-12-31.AR.nc: ['latitude', 'longitude', 'time', 'lon', 'dayofyear', 'precip']
Variables in chirps_precip_daily.1981-01-01.2023-12-31.TN.nc: ['latitude', 'longitude', 'time', 'lon', 'dayofyear', 'precip']
Variables in chirps_precip_daily.1981-01-01.2023-12-31.OH.nc: ['latitude', 'longitude', 'time', 'lon', 'dayofyear', 'precip']
Variables in chirps_precip_daily.1981-01-01.2023-12-31.MT.nc: ['latitude', 'longitude', 'time', 'lon', 'dayofyear', 'p

In [ ]:
def convert_times(cftime_array):
    # Convert cftime.DatetimeGregorian objects to regular datetime objects
    return [pd.Timestamp(str(date)) for date in cftime_array]

def process_file(file_path, state_abbr):
    with nc.Dataset(file_path, 'r') as ds:
        precip_data = ds['precip'][:]
        avg_precip_data = np.nanmean(precip_data, axis=(1, 2))
        avg_precip_data_rounded = np.round(avg_precip_data, 2)
        time_dim = ds.variables['time']
        dates = convert_times(nc.num2date(time_dim[:], units=time_dim.units))
        precip_series = pd.Series(avg_precip_data_rounded, index=dates)
        monthly_avg_precip = precip_series.resample('M').mean()

        # Rename column from 0 to AVG PRECIP ANOM
        monthly_avg_precip.name = "AVG PRECIP ANOM"

        # Save the monthly average precipitation anomalies to a CSV file
        output_filename = f"monthly_avg_precip_{state_abbr}.csv"
        monthly_avg_precip.to_csv(os.path.join(directory, output_filename))





In [ ]:
# Loop
for filename in os.listdir(directory):
    if filename.startswith('chirps'):
        file_path = os.path.join(directory, filename)
        # Sort by state abbreviation in file
        state_abbr = filename.split('.')[-2].upper()
        process_file(file_path, state_abbr)

In [ ]:
#ENSO DATA TRIMMING
enso_df = enso_df[(enso_df['YEAR'] >= 1981) & (enso_df['YEAR'] <= 2020)]

In [ ]:
# Function to process each state's precipitation file and merge with ENSO data
def merge_precip_with_enso(state_abbr, enso_df):
    precip_file = os.path.join(directory, f'monthly_avg_precip_{state_abbr}.csv')
    precip_df = pd.read_csv(precip_file)

    # Convert index to YEAR and Month columns
    precip_df['YEAR'] = pd.to_datetime(precip_df['Unnamed: 0']).dt.year
    precip_df['Month'] = pd.to_datetime(precip_df['Unnamed: 0']).dt.month_name().str.upper()

    # Pivot to have months as columns
    precip_pivot = precip_df.pivot(index='YEAR', columns='Month', values='AVG PRECIP ANOM')

    # Merge with ENSO data
    merged_df = enso_df.merge(precip_pivot, on='YEAR', how='left')
    merged_df.to_csv(os.path.join(directory, f'ENSO_with_precip_{state_abbr}.csv'), index=False)

# Process and merge for each state
for filename in os.listdir(directory):
    if filename.startswith('monthly_avg_precip_') and filename.endswith('.csv'):
        state_abbr = filename.split('_')[-1].split('.')[0].upper()
        merge_precip_with_enso(state_abbr, enso_df)

In [ ]:
# Load the ENSO data and trim to the years 1981-2020
enso_df = pd.read_csv(os.path.join(directory, 'ENSO.csv'))
enso_df.rename(columns={enso_df.columns[0]: 'YEAR'}, inplace=True)
enso_df = enso_df[(enso_df['YEAR'] >= 1981) & (enso_df['YEAR'] <= 2020)]

# Function to process and merge each state's precipitation file
def merge_state_precip(state_file, enso_df):
    state_precip = pd.read_csv(state_file)
    state_precip['YEAR'] = pd.to_datetime(state_precip['Unnamed: 0']).dt.year
    state_precip['Month'] = pd.to_datetime(state_precip['Unnamed: 0']).dt.month_name().str.upper()
    state_precip_pivot = state_precip.pivot(index='YEAR', columns='Month', values='AVG PRECIP ANOM')
    state_abbr = state_file.split('_')[-1].split('.')[0].upper()
    state_precip_pivot.columns = [f"{col}_{state_abbr}" for col in state_precip_pivot.columns]
    return enso_df.merge(state_precip_pivot, on='YEAR', how='left')

# Loop through all state files and merge
for file in os.listdir(directory):
    if file.startswith('monthly_avg_precip_') and file.endswith('.csv'):
        enso_df = merge_state_precip(os.path.join(directory, file), enso_df)

# Print or save the combined DataFrame
print(enso_df.head())  # Print the first few rows
enso_df.to_csv(os.path.join(directory, 'combined_ENSO_highresprecip.csv'), index=False)  # Save to a CSV file

   YEAR   JAN   FEB   MAR   APR   MAY   JUN   JUL   AUG   SEP  ...  \
0  1981 -0.36 -0.64 -0.64 -0.53 -0.57 -0.46 -0.64 -0.53 -0.19  ...   
1  1982  0.13 -0.17  0.13  0.21  0.45  0.53  0.37  0.73  1.49  ...   
2  1983  2.35  1.94  1.38  0.95  0.90  0.54 -0.11 -0.27 -0.52  ...   
3  1984 -0.67 -0.19 -0.52 -0.68 -0.73 -0.90 -0.50 -0.24 -0.34  ...   
4  1985 -1.16 -0.72 -0.79 -1.18 -1.03 -0.91 -0.74 -0.56 -0.70  ...   

   DECEMBER_SD  FEBRUARY_SD  JANUARY_SD   JULY_SD   JUNE_SD  MARCH_SD  \
0    -0.028065    -0.090000   -0.230645  0.525161 -0.562667 -0.055806   
1    -0.077742    -0.193929    0.171613  0.289032 -0.610667  0.297419   
2    -0.052581    -0.286071   -0.157742  0.103226  0.191000  0.557097   
3     0.027097    -0.184828   -0.139032 -0.563548  1.981000 -0.054516   
4    -0.053871    -0.276429   -0.103548 -0.592903 -1.062000  0.122258   

     MAY_SD  NOVEMBER_SD  OCTOBER_SD  SEPTEMBER_SD  
0 -0.553548    -0.060333   -0.007419     -0.588667  
1  2.291935     0.031667    1.6083

In [ ]:
enso_df = pd.read_csv(os.path.join(directory, 'ENSO.csv'))
enso_melted = enso_df.melt(id_vars='YEAR', var_name='Month', value_name='ENSO_Value')
enso_melted['Year-Month'] = enso_melted['YEAR'].astype(str) + '-' + enso_melted['Month']
enso_melted['Year-Month'] = pd.to_datetime(enso_melted['Year-Month']).dt.to_period('M')
enso_filtered = enso_melted[(enso_melted['Year-Month'].dt.year >= 1981) & (enso_melted['Year-Month'].dt.year <= 2020)]

def process_precip_data(state_file):
    state_precip = pd.read_csv(state_file)
    state_precip['Year-Month'] = pd.to_datetime(state_precip['Unnamed: 0']).dt.to_period('M')
    state_abbr = state_file.split('_')[-1].split('.')[0].upper()
    state_precip.rename(columns={'AVG PRECIP ANOM': f'Precip_Anomaly_{state_abbr}'}, inplace=True)
    return state_precip[['Year-Month', f'Precip_Anomaly_{state_abbr}']]

all_states_precip = pd.DataFrame()
for file in os.listdir(directory):
    if file.startswith('monthly_avg_precip_') and file.endswith('.csv'):
        state_data = process_precip_data(os.path.join(directory, file))
        if all_states_precip.empty:
            all_states_precip = state_data
        else:
            all_states_precip = all_states_precip.merge(state_data, on='Year-Month', how='outer')

# Merge ENSO data
combined_df = enso_filtered.merge(all_states_precip, on='Year-Month', how='left').sort_values(by='Year-Month')

# Print or save the combined DataFrame
print(combined_df.head())
combined_df.to_csv(os.path.join(directory, 'combined_ENSO_highresprecip.csv'), index=False)

/var/folders/nh/sgqb22xd02q8jwdk9cq535fh0000gn/T/ipykernel_46162/2432207104.py:4: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  enso_melted['Year-Month'] = pd.to_datetime(enso_melted['Year-Month']).dt.to_period('M')


     YEAR Month  ENSO_Value Year-Month  Precip_Anomaly_ID  Precip_Anomaly_KY  \
0    1981   JAN       -0.36    1981-01          -0.629677          -2.257742   
40   1981   FEB       -0.64    1981-02           0.056429           0.086071   
80   1981   MAR       -0.64    1981-03          -0.217742          -1.000645   
120  1981   APR       -0.53    1981-04           0.116000           0.275000   
160  1981   MAY       -0.57    1981-05           0.571935           0.944516   

     Precip_Anomaly_DE  Precip_Anomaly_FL  Precip_Anomaly_PA  \
0            -1.884516          -1.727742          -1.742581   
40            0.187143           1.146429           2.024643   
80           -1.438387          -0.646452          -1.348710   
120           0.523000          -2.041000           0.703667   
160           0.661935          -1.170645          -0.683548   

     Precip_Anomaly_RI  ...  Precip_Anomaly_NM  Precip_Anomaly_OK  \
0            -2.242258  ...          -0.118387          -1.003226

In [ ]:
enso_df = pd.read_csv(os.path.join(directory, 'ENSO.csv'))
enso_melted = enso_df.melt(id_vars='YEAR', var_name='Month', value_name='ENSO_Value')
enso_melted['Year-Month'] = enso_melted['YEAR'].astype(str) + '-' + enso_melted['Month']
enso_melted['Year-Month'] = pd.to_datetime(enso_melted['Year-Month']).dt.to_period('M')
enso_filtered = enso_melted[(enso_melted['Year-Month'].dt.year >= 1981) & (enso_melted['Year-Month'].dt.year <= 2020)]

def process_precip_data(state_file):
    state_precip = pd.read_csv(state_file)
    state_precip['Year-Month'] = pd.to_datetime(state_precip['Unnamed: 0']).dt.to_period('M')
    state_abbr = state_file.split('_')[-1].split('.')[0].upper()
    state_precip.rename(columns={'AVG PRECIP ANOM': f'Precip_Anomaly_{state_abbr}'}, inplace=True)
    return state_precip[['Year-Month', f'Precip_Anomaly_{state_abbr}']]

all_states_precip = pd.DataFrame()
for file in os.listdir(directory):
    if file.startswith('monthly_avg_precip_') and file.endswith('.csv'):
        state_data = process_precip_data(os.path.join(directory, file))
        if all_states_precip.empty:
            all_states_precip = state_data
        else:
            all_states_precip = all_states_precip.merge(state_data, on='Year-Month', how='outer')

# Merge ENSO data
combined_df = enso_filtered.merge(all_states_precip, on='Year-Month', how='left').sort_values(by='Year-Month')

# Round relevant numeric columns to 2 decimal places
columns_to_round = ['ENSO_Value'] + [col for col in combined_df.columns if col.startswith('Precip_Anomaly_')]
combined_df[columns_to_round] = combined_df[columns_to_round].round(2)

# Print or save the combined DataFrame
print(combined_df.head())
combined_df.to_csv(os.path.join(directory, 'combined_ENSO_highresprecip.csv'), index=False)

/var/folders/nh/sgqb22xd02q8jwdk9cq535fh0000gn/T/ipykernel_46162/1702027022.py:4: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  enso_melted['Year-Month'] = pd.to_datetime(enso_melted['Year-Month']).dt.to_period('M')


     YEAR Month  ENSO_Value Year-Month  Precip_Anomaly_ID  Precip_Anomaly_KY  \
0    1981   JAN       -0.36    1981-01              -0.63              -2.26   
40   1981   FEB       -0.64    1981-02               0.06               0.09   
80   1981   MAR       -0.64    1981-03              -0.22              -1.00   
120  1981   APR       -0.53    1981-04               0.12               0.28   
160  1981   MAY       -0.57    1981-05               0.57               0.94   

     Precip_Anomaly_DE  Precip_Anomaly_FL  Precip_Anomaly_PA  \
0                -1.88              -1.73              -1.74   
40                0.19               1.15               2.02   
80               -1.44              -0.65              -1.35   
120               0.52              -2.04               0.70   
160               0.66              -1.17              -0.68   

     Precip_Anomaly_RI  ...  Precip_Anomaly_NM  Precip_Anomaly_OK  \
0                -2.24  ...              -0.12              -1.00